In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

data = [
    ("S101", "Biscuits", "Food", 120, "Tasty Biscuits [10% off]"),
    ("S102", "Shampoo", "Hygiene", 85, "Smoothens Hair [5% off]"),
    ("S103", "Banana", "Food", 150, "Fresh Bananas"),
    ("S101", "Toothpaste", "Hygiene", 300, "Protects Teeth"),
    ("S102", "Shirt", "Clothes", 65, "Cotton Shirts [20% off]"),
]

columns = ["StoreID", "ProductName", "Category", "SoldUnits", "Description"]

df = spark.createDataFrame(data, schema=columns)

df.show(truncate=False)

+-------+-----------+--------+---------+------------------------+
|StoreID|ProductName|Category|SoldUnits|Description             |
+-------+-----------+--------+---------+------------------------+
|S101   |Biscuits   |Food    |120      |Tasty Biscuits [10% off]|
|S102   |Shampoo    |Hygiene |85       |Smoothens Hair [5% off] |
|S103   |Banana     |Food    |150      |Fresh Bananas           |
|S101   |Toothpaste |Hygiene |300      |Protects Teeth          |
|S102   |Shirt      |Clothes |65       |Cotton Shirts [20% off] |
+-------+-----------+--------+---------+------------------------+



In [41]:
df.withColumn("converted_arr", split(col("Description"), " ")[2]).withColumn(
    "Discount",
    when(
        regexp_replace(col("converted_arr"), r"[\[\]%]", "").isNotNull(),
        regexp_replace(col("converted_arr"), r"[\[\]%]", "") / 100,
    ).otherwise(lit("0.0")),
).select(
    "Category", "Description", "Discount", "ProductName", "SoldUnits", "StoreID"
).show()

+--------+--------------------+--------+-----------+---------+-------+
|Category|         Description|Discount|ProductName|SoldUnits|StoreID|
+--------+--------------------+--------+-----------+---------+-------+
|    Food|Tasty Biscuits [1...|     0.1|   Biscuits|      120|   S101|
| Hygiene|Smoothens Hair [5...|    0.05|    Shampoo|       85|   S102|
|    Food|       Fresh Bananas|     0.0|     Banana|      150|   S103|
| Hygiene|      Protects Teeth|     0.0| Toothpaste|      300|   S101|
| Clothes|Cotton Shirts [20...|     0.2|      Shirt|       65|   S102|
+--------+--------------------+--------+-----------+---------+-------+

